In [1]:
L = 5
lattice = (L, L)
p = 0.4
#generating 
x = rand(L^2)
y = x .> p 
M = zeros(L, L)
for i in 1:length(M)
    M[i] = y[i]
end

M

5×5 Matrix{Float64}:
 1.0  0.0  1.0  0.0  0.0
 0.0  0.0  0.0  0.0  1.0
 0.0  1.0  1.0  1.0  1.0
 1.0  0.0  1.0  0.0  0.0
 0.0  1.0  0.0  1.0  1.0

In [2]:
function neighbors2d(I::Tuple{Int,Int})
    Lx, Ly = Dims(I)
    N = Lx*Ly #Total number of elements
    neigh = [zeros(Int,4) for i in 1:N] #initialization
    for i in 1:Ly #running thorugh a row
        for j in 1:Lx #running through a coloumn
            N = mod1(j-1,Lx) + Lx * (i - 1)
            S = mod1(j+1,Lx) + Lx * (i - 1)
            E = j + Lx * (mod1(i+1,Ly)-1)
            W = j + Lx * (mod1(i-1,Ly)-1)
            neigh[j + Lx * (i-1)] = [N,W,S,E]
            #println("The $(j + Lx * (i-1)) position has as neighbours: N=$N, W=$W, S=$S, E=$E")
        end
    end
    return neigh
end#count clusters
function count_clusters(M)
    Lx, Ly = size(M)
    N = Lx * Ly
    neigh = neighbors2d(size(M))
    visited = falses(N)
    cluster_count = 0
    cluster_size = Int[]
    for i in 1:N
        if M[i] == 1 && !visited[i]
            cluster_count += 1
            # Perform DFS or BFS to mark all connected sites
            stack = [i]
            current_cluster_size = 0
            while !isempty(stack)
                current = pop!(stack)
                if !visited[current]
                    visited[current] = true
                    current_cluster_size += 1
                    for neighbor in neigh[current]
                        if M[neighbor] == 1 && !visited[neighbor]
                            push!(stack, neighbor)
                        end
                    end
                end
            end
            push!(cluster_size, current_cluster_size)
        end
    end
    if cluster_count == 0
        return (0, 0)
    else
        return cluster_count, cluster_size
    end
end

num_clusters, cluster_sizes = count_clusters(M)



(6, [1, 1, 6, 1, 1, 2])

In [ ]:
#show percolation in different p
using Plots
L = 100
p_values = (0.1, 0.4, 0.55, 0.62)
heatmap(size=(800, 800))
for p in p_values
    #generating 
    x = rand(L^2)
    y = x .< p 
    M = zeros(L, L)
    for i in 1:length(M)
        M[i] = y[i]
    end
    a, b = count_clusters(M)
    println("For p = $p, number of clusters: $a, largest cluster size fraction: $(maximum(b) / L^2)")
    heatmap(M, c=:blues, xlabel="X", ylabel="Y", title="Percolation Lattice p=$p", aspect_ratio=1)
end
display(current())

In [ ]:
using Plots

L = 100
p_values = (0.1, 0.4, 0.55, 0.62)

plots = []
for p in p_values
    M = reshape(rand(L^2) .< p, L, L)    # matrice booleana 0/1
    push!(plots,
        heatmap(
            -M,
            c = :grays,            # scala bianco -> nero
            colorbar = false,
            legend = false,
            xticks = false,
            yticks = false,
            framestyle = :none,
            aspect_ratio = 1
        )
    )
end

plt = plot(plots..., layout = (1,4), size = (1200, 300), spacing = 0)
display(plt)

In [ ]:
#Evolution of biggest cluster with p
#Benchmarking
using BenchmarkTools
plt = plot()
Nsim = 1000
p_values = 0.3:0.02:0.7
average_biggest_clusters = zeros(length(p_values))
biggest_clusters = zeros(length(p_values))

for L in (20, 50, 100, 200)
    M = zeros(L, L)
    average_biggest_clusters .= 0.0
    for sim in 1:Nsim    
        biggest_clusters .= 0.0
        
        i = 1
        for p in p_values
            #generating
            x = rand(L^2)
            y = x .<= p
            M .= 0
            M = reshape(y, (L, L))
            a, b = count_clusters(M)
            biggest_clusters[i] = maximum(b) / L / L
            i += 1
        end
        average_biggest_clusters += biggest_clusters 
    end
    average_biggest_clusters ./= (Nsim)
    plot!(p_values, average_biggest_clusters, xlabel="Occupation Probability p", ylabel="Size of Biggest Cluster", title="Percolation Transition", legend=true, label="L = $L")
end
display(plt)

In [ ]:
x0 = rand()
r = 3.6
N = 100
mappa = let x=x0;
    vcat(x0,[x = r * x * (1-x) for _ in 1:N])
end

In [ ]:
plot(mappa)

In [ ]:
using Plots

function diagramma_biforcazione_log(s,f)
    # Generiamo 1000 valori di r da 0 a 4.0
    r_valori = range(s,f, length=1000)
    x = fill(0.5, length(r_valori)) # Partiamo da x=0.5 per tutti
    
    # 1. Assestamento (Burn-in): iteriamo 500 volte senza disegnare per raggiungere l'equilibrio
    for _ in 1:500
        x .= r_valori .* x .* (1 .- x)
    end
    
    p_bif = plot(title="Diagramma di Biforcazione", xlabel="Tasso di crescita (r)", ylabel="Valori finali di x", legend=false, size=(800, 500))
    
    # 2. Registriamo i comportamenti: disegniamo le successive 150 iterazioni
    for _ in 1:150
        x .= r_valori .* x .* (1 .- x)
        # alpha=0.05 rende i punti trasparenti per vedere la densità!
        scatter!(p_bif, r_valori, x, markersize=1, markerstrokewidth=0, color=:black, alpha=0.05)
    end
    
    return p_bif
end

diagramma_biforcazione_log(0, 4) # Avvia e mostra il diagramma
vline!([3.569])


In [ ]:
diagramma_biforcazione_log(3.5, 3.8)

In [ ]:
using Roots
using Printf
#definisco la mappa 
f(x, r) = r * x * (1-x)

function iterate_mappa(x, r, n)
    for _ in 1:n
        x = f(x, r)
    end
    return x
end

#Per x0 = 0.5 orbita passa sempre!

function closeness(n, r_guess)
    period = 2^n
    g(r) = iterate_mappa(0.5, r, period) - 0.5
    return find_zero(g, r_guess)
end

function calc(n)
    r_vals = zeros(n)
    d_vals = zeros(n)
    guesses = [3.0, 3.449, 3.54, 3.564, 3.57]
    r_star = 3.56991

    println("n \t r_n \t \t d_n \t \t δ \t \t a")
    println("-"^60)

    for i in 1:n
        r_vals[i] = closeness(i, guesses[i])
        x = iterate_mappa(0.5, r_vals[i], 2^(i-1))
        d_vals[i] = x - 0.5
        
        if i > 2
            delta = (r_vals[i-2] - r_vals[i-1]) ./ (r_vals[i-1] - r_vals[i])
            alpha = d_vals[i-1] / d_vals[i]
            
            @printf("%d \t %.5f \t %.6f \t %.6f \t %.5f\n", i, r_vals[i], d_vals[i], delta, alpha)
        elseif i > 1
            alpha = d_vals[i-1] / d_vals[i]
            @printf("%d \t %.5f \t %.6f \t ---  \t \t %.5f\n", i, r_vals[i], d_vals[i],  alpha)
        else
            @printf("%d \t %.5f \t %.6f \t --- \t \t ---\n", i, r_vals[i], d_vals[i])
        end
    end
end
calc(5)

In [ ]:
using Plots

function diagramma_biforcazione_sin(s,f)
    # Generiamo 1000 valori di r da 0 a 4.0
    r_valori = range(s,f, length=1000)
    x = fill(0.5, length(r_valori)) # Partiamo da x=0.5 per tutti
    
    # 1. Assestamento (Burn-in): iteriamo 500 volte senza disegnare per raggiungere l'equilibrio
    for _ in 1:500
        x .= r_valori .* sin.(pi * x)
    end
    
    p_bif = plot(title="Diagramma di Biforcazione", xlabel="Tasso di crescita (r)", ylabel="Valori finali di x", legend=false, size=(800, 500))
    
    # 2. Registriamo i comportamenti: disegniamo le successive 150 iterazioni
    for _ in 1:150
        x .= r_valori .* sin.(pi * x)
        # alpha=0.05 rende i punti trasparenti per vedere la densità!
        scatter!(p_bif, r_valori, x, markersize=1, markerstrokewidth=0, color=:black, alpha=0.05)
    end
    
    return p_bif
end

diagramma_biforcazione_sin(0, 1) # Avvia e mostra il diagramma
vline!([0.86539])


In [ ]:
using Roots

#definisco la mappa 
f(x, r) = r * sin(pi*x)

function iterate_mappa(x, r, n)
    for _ in 1:n
        x = f(x, r)
    end
    return x
end

#Per x0 = 0.5 orbita passa sempre!

function closeness(n, r_guess)
    period = 2^n
    g(r) = iterate_mappa(0.5, r, period) - 0.5
    return find_zero(g, r_guess)
end

function calc(n)
    r_vals = zeros(n)
    d_vals = zeros(n)
    guesses = [0.718, 0.832, 0.858, 0.8635, 0.8652]
    r_star = 3.56991

    println("n \t r_n \t \t d_n \t \t δ \t \t a")
    println("-"^60)

    for i in 1:n
        r_vals[i] = closeness(i, guesses[i])
        x = iterate_mappa(0.5, r_vals[i], 2^(i-1))
        d_vals[i] = x - 0.5
        
        if i > 2
            delta = (r_vals[i-2] - r_vals[i-1]) ./ (r_vals[i-1] - r_vals[i])
            alpha = d_vals[i-1] / d_vals[i]
            
            @printf("%d \t %.5f \t %.6f \t %.6f \t %.5f\n", i, r_vals[i], d_vals[i], delta, alpha)
        elseif i > 1
            alpha = d_vals[i-1] / d_vals[i]
            @printf("%d \t %.5f \t %.6f \t ---  \t \t %.5f\n", i, r_vals[i], d_vals[i],  alpha)
        else
            @printf("%d \t %.5f \t %.6f \t --- \t \t ---\n", i, r_vals[i], d_vals[i])
        end
    end
end
calc(5)

In [ ]:
using Plots

function diagramma_biforcazione_sin(s,f)
    # Generiamo 1000 valori di r da 0 a 4.0
    r_valori = range(s,f, length=1000)
    x = fill(0.5, length(r_valori)) # Partiamo da x=0.5 per tutti
    
    # 1. Assestamento (Burn-in): iteriamo 500 volte senza disegnare per raggiungere l'equilibrio
    for _ in 1:500
        x .= r_valori .* cos.(pi * x)
    end
    
    p_bif = plot(title="Diagramma di Biforcazione", xlabel="Tasso di crescita (r)", ylabel="Valori finali di x", legend=false, size=(800, 500))
    
    # 2. Registriamo i comportamenti: disegniamo le successive 150 iterazioni
    for _ in 1:300
        x .= r_valori .* cos.(pi * x)
        # alpha=0.05 rende i punti trasparenti per vedere la densità!
        scatter!(p_bif, r_valori, x, markersize=1, markerstrokewidth=0, color=:black, alpha=0.05)
    end
    
    return p_bif
end

diagramma_biforcazione_sin(0, 1) # Avvia e mostra il diagramma

